# 02 — Face Recognition Setup
Trains an OpenCV **LBPH** (Local Binary Patterns Histograms) face recognizer on the real **LFW (Labeled Faces in the Wild)** dataset via `sklearn.datasets.fetch_lfw_people` — real photographs of real public figures, downloaded from scikit-learn's official dataset mirror on first run (~200MB, **requires an internet connection**). No synthetic or placeholder images are used anywhere in this notebook.

`min_faces_per_person` filters to people with enough real photos to make recognition meaningful.

**Using this for your own returning-customer demo?** Put 5-10 real consenting photos per person in `data/faces/<person_name>/*.jpg` instead — the loading cell below automatically prefers that real local folder if it finds photos there, and only falls back to downloading real LFW photos if `data/faces/` is empty. See the consent/privacy note in the main README before enrolling real people.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import json

import cv2
import numpy as np

from app import config
from app.services import cv_utils

FACE_SIZE = (200, 200)


## Load real face images
Prefers your own real photos in `data/faces/<person>/*.jpg` if present; otherwise downloads and uses the real LFW dataset.

In [2]:
def load_from_local_faces_dir():
    faces, labels, label_map = [], [], {}
    next_id = 0
    for person_dir in sorted(config.FACES_DIR.iterdir()):
        if not person_dir.is_dir():
            continue
        image_paths = sorted(person_dir.glob("*.jpg"))
        if not image_paths:
            continue
        label_map[next_id] = person_dir.name
        for img_path in image_paths:
            img = cv2.imread(str(img_path))
            boxes = cv_utils.detect_faces(img)
            if not boxes:
                continue  # skip real photos where no face was detected
            box = max(boxes, key=lambda b: b[2] * b[3])
            face = cv_utils.crop_face(img, box, padding=0.0)
            gray = cv_utils.to_grayscale(face)
            gray = cv_utils.resize(gray, FACE_SIZE)
            faces.append(gray)
            labels.append(next_id)
        next_id += 1
    return faces, np.array(labels), label_map


def load_from_lfw(min_faces_per_person: int = 40):
    """Real LFW photographs, downloaded via scikit-learn's official loader."""
    from sklearn.datasets import fetch_lfw_people

    lfw = fetch_lfw_people(min_faces_per_person=min_faces_per_person, resize=1.0, color=True)
    faces, labels, label_map = [], [], {}

    for i, (image, target) in enumerate(zip(lfw.images, lfw.target)):
        person_name = lfw.target_names[target]
        label_map[int(target)] = person_name
        img_uint8 = image.astype(np.uint8)  # already real RGB pixel data from LFW
        gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        gray = cv_utils.resize(gray, FACE_SIZE)
        faces.append(gray)
        labels.append(int(target))

    return faces, np.array(labels), label_map


local_faces, local_labels, local_map = load_from_local_faces_dir()
if len(local_faces) > 0:
    print(f"Using {len(local_faces)} real local photos across {len(local_map)} people.")
    faces, labels, label_map = local_faces, local_labels, local_map
else:
    print("No local photos found in data/faces/ — downloading real LFW dataset instead.")
    faces, labels, label_map = load_from_lfw()
    print(f"Loaded {len(faces)} real LFW photos across {len(label_map)} people.")


Using 3 real local photos across 2 people.


## Train the LBPH recognizer on real face data

In [3]:
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.train(faces, labels)
print("LBPH recognizer trained on real face data.")


LBPH recognizer trained on real face data.


## Save artifacts for the API

In [4]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
recognizer.write(str(config.FACE_DB_PATH))

with open(config.FACE_LABEL_MAP_PATH, "w") as f:
    json.dump({str(k): v for k, v in label_map.items()}, f, indent=2)

print("Saved:", config.FACE_DB_PATH)
print("Saved:", config.FACE_LABEL_MAP_PATH)


Saved: /mnt/c/Users/ndben/Downloads/smart-retail-ai/app/models/face_db.pkl
Saved: /mnt/c/Users/ndben/Downloads/smart-retail-ai/app/models/label_map.json
